# 5.4. Numerical Stability and Initialization
D2L의 Numerical Stability and Initialization장을 PyTorch 기준으로 정리함.

1. Vanishing Gradient
2. Exploding Gradient
3. Symmetry 문제
4. Weight Initialization
5. Xavier Initialization

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 깊은 신경망과 Gradient

예를 들어서 이런 신경망이 있다고 하자.

```text
X
↓
Layer 1
↓
Layer 2
↓
Layer 3
↓
Output
↓
Loss
```

순전파에서는 앞에서 뒤로 계산한다.

    X -> Layer1 -> Layer2 -> Layer3 -> output -> Loss

역전파에서는 반대로 gradient가 전달된다.

    Loss -> Layer3 gradient -> Layer2 gradient -> Layer1 gradient

이때 미분값들이 계속 곱해진다.

지난 장의 Chain Rule이 여기서 나온다.

$$
x \rightarrow h_1 \rightarrow h_2 \rightarrow y \rightarrow L
$$

이면

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\times\frac{\partial y}{\partial h_2}\times\frac{\partial h_2}{\partial h_1}\times\frac{\partial h_1}{\partial x}
$$

이다. D2L에서도 같은 네트워크의 gradient가 여러 층의 미분 행렬을 연속해서 곱한 형태가 되기 때문에 결과가 매우 커지거나 매우 작아질 수 있다고 설명한다.

예를 들어
```text
각 층의 미분값 = 0.5

Layer가 3개라면

0.5 x 0.5 x 0.5 = 0.125

Layer 10개라면

0.5^10 = 0.000977

반대로 각 층에서 2가 곱해진다면

2^10 = 1024 가 된다.
```

    작은 값을 계속 곱하면 0에 가까워지고
    큰 값을 계속 곱하면 엄청 커진다.


## 2. Vanishing Gradient

Vanishing Gradient는 역전파 과정에서 gradient가 계속 작어져서 앞쪽 Layer까지 거의 전달되지 않는 문제이다.

예를 들어서 Layer를 지나며 gradient가 0.5가 곱해진다고 했을때

```text
1
↓ × 0.5
0.5
↓ × 0.5
0.25
↓ × 0.5
0.125
↓ × 0.5
0.0625
```

Layer가 많아질수록 gradient가 0에 가까워진다.

그러면 앞쪽 Layer의 weight가 거의 수정되지 않는다.

    W_new = W_old - learning_rate x gradient

gradient가 거의 0이면

    W_new ≈ W_old

이게 중요한 이유는 optimizer 때문이다.

예를 들어서

```text
learning_rate = 0.01
gradient = 0.000001

이면

weight 변화량
= 0.01 x 0.000001 = 0.00000001
```

D2L에서는 특히 sigmoid가 입력 절댓값이 커질수록 gradient가 매우 작아지는 성질 때문에 깊은 신경망에서 vanishing gradient를 일으킬 수 있다고 설명하고, 이게 ReLU가 널리 사용되는 이유 중 하나라고 설명한다.

In [2]:
x = torch.tensor(1.0, requires_grad=True)

y = x

for _ in range(10):
    y = y * 0.5

y.backward()

print("y:", y.item())
print("x의 gradient:", x.grad.item())

y: 0.0009765625
x의 gradient: 0.0009765625


## 3. Exploding Gradient

Exploding Gradient는 역전파 과정에서 gradient가 계속 커지는 문제이다.

예를 들어서 각 Layer를 지나며 2가 곱해진다고 했을때

```text
1
↓ × 2
2
↓ × 2
4
↓ × 2
8
↓ × 2
16
↓ × 2
32
```

10번 반복하면 2^10 = 1024가 된다.

gradient가 지나치게 커질 수 있다.

그러면 optimizer가 weight를 너무 크게 수정하게 된다.

W가 한번에 크게 움직이며 학습이 불안정하게 된다.

D2L은 무작위 행렬을 반복해서 100번 곱하는 예제로 값이 $10^{23}\sim10^{25}$ 수준까지 폭발할 수 있음을 보여 준다. 깊은 신경망의 gradient 역시 여러 행렬의 곱으로 계산되기 때문에 같은 종류의 문제가 발생할 수 있다는 설명입니다.


In [3]:
x = torch.tensor(1.0, requires_grad=True)

y = x

for _ in range(10):
    y = y * 2

y.backward()

print("y:", y.item())
print("x의 gradient:", x.grad.item())

y: 1024.0
x의 gradient: 1024.0


## 4. Symmetry 문제

그러면 간단히 모든 weight를 0으로 시작하면 되지 않나

MLP에선 문제가 발생한다.

예를 들어서 hidden neuron이 3개 있다고 해보자

```text
Neuron 1
Neuron 2
Neuron 3
```

세 뉴런의 weight를 모두 똑같은 값으로 초기화 했다고 했을때

세 뉴런은 같은 입력을 받아서, 같은 weight를 가지고, 같은 출력을 만들고, 역전파에서 같은 gradient를 받는다.

결국 업데이트 후에도 계속 같은 값을 가지게 된다. 이러면 뉴런이 여러 개인 것이 의미가 없다.

사실상 하나의 뉴런을 여러 번 복사한 것과 같다. 이런 문제를 Symmetry라고 한다.

그래서 신경망의 weight는 일반적으로 서로 다른 random 값으로 초기화한다.

D2L의 예에서도 hidden unit들을 같은 상수로 초기화하면 순전파 결과도 같고 역전파 gradient도 같아져, 계속 동일하게 움직인다.  
결과적으로 여러 hidden unit의 표현력을 제대로 사용할 수 없게 된다.

```text
weight -> random initialization

bias -> 0으로 초기화 가능
```

결론은 뉴런의 weight는 처음부터 서로 다르게 시작해야 한다는 것이다.

## 5. Parameter Initialization

신경망 학습을 시작하기 전에는 아직 학습된 weight가 존재하지 않는다.

그래서 처음 시작할 weight 값을 정해야 한다. 이걸 Parameter Initialization이라고 한다. 

가능하다면

```text
순전파의 값이 너무 커지거나 작아지지 않고
역전파의 gradient가 너무 커지거나 작아지지 않도록
초기값의 크기를 조절해야 한다.
```

```text
너무 작은 weight -> 값과 gradient가 작아질 위험

너무 큰 weight -> 값과 gradient가 커질 위험

적절한 크기의 random weight → 안정적인 학습
```

D2L에서는 gradient 및 parameter 크기를 안정적으로 유지하기 위한 주요 방법 중 하나로 careful initialization을 제시한다.


## 6. Xavier Initialization

이 부분은 수식보다는 왜 이걸 하는가를 이해하는게 먼저라고 한다.

Xavier Initialization의 목적은 Layer를 통과할 때 값의 크기가 갑자기 커지거나 작아지는 것을 줄이는 것이다.

Linear Layer를 생각해볼때

```text
입력 x -> Wx + b -> 출력
```

입력 feature가 많아질수록 많은 값들이 더해진다.

예를 들어서 

    nn.Linear(784, 256)
이면 하나의 뉴런은 784개의 입력을 사용한다.

그래서 W를 아무 크기로 random하게 만들면 입력 개수에 따라 출력값의 크기가 지나치게 커지거나 작아질 수 있다.

Xavier Initialization은 입력 뉴런 수와 출력 뉴런 수를 고려해서 weight의 초기 범위를 결정한다.

D2L에서는 fully connected layer의

$$
o_i=\sum_{j=1}^{n_{in}}w_{ij}x_j
$$

를 생각한다. 입력과 weight가 서로 독립이고 평균이 0이라는 가정 아래 출력의 분산은

$$
\operatorname{Var}(o_i)
=
n_{in}\sigma^2\gamma^2
$$

가 된다. 그래서 입력 수가 많아진다고 출력 분산이 계속 커지지 않도록 weight 분산을 조절할 필요가 있다. 역전파에서도 비슷하게 출력 수가 gradient 분산에 영향을 주기 때문에 Xavier는 양쪽 조건을 절충한다.

결과적으로 이것을 쓴다.
$$
\sigma^2
=
\frac{2}{n_{in}+n_{out}}
$$

지금은 이렇게 생각하면 된다.
```text
Xavier Initialization

입력 뉴런 수
+
출력 뉴런 수

를 고려해서

weight가 너무 크지도
너무 작지도 않게

초기 random 값의 크기를 결정하는 방법
```

In [4]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

In [ ]:
for layer in model:
    if isinstance(layer, nn.Linear):
        nn.init.xavier_uniform_(layer.weight) # Linear Layer의 w를 Xavier방식으로 초기화
        nn.init.zeros_(layer.bias)            # bias 0으로 초기화

D2L의 이 장은 Xavier를 중심으로 초기화 원리를 설명하며, 실제 딥러닝 프레임워크에는 상황별로 다양한 초기화 방법이 제공된다고 설명한다.


## 7. 오늘의 정리

- 깊은 신경망에서는 역전파할 때 여러 미분값이 계속 곱해진다.
- 작은 미분값이 계속 곱해지면 gradient가 0에 가까워지는 Vanishing Gradient가 발생할 수 있다.
- gradient가 너무 작으면 optimizer가 weight를 거의 수정하지 못한다.
- 큰 미분값이 계속 곱해지면 gradient가 매우 커지는 Exploding Gradient가 발생할 수 있다.
- gradient가 너무 크면 weight가 한 번에 너무 크게 변하면서 학습이 불안정해질 수 있다.
- Sigmoid는 입력이 특정 영역을 벗어나면 gradient가 작아져 Vanishing Gradient가 발생하기 쉽다.
- ReLU는 이러한 문제를 완화하기 때문에 깊은 신경망에서 많이 사용된다.
- MLP의 모든 weight를 같은 값으로 초기화하면 뉴런들이 똑같이 학습하는 Symmetry 문제가 발생한다.
- 따라서 weight는 서로 다른 random 값으로 초기화해야 한다.
- Xavier Initialization은 입력 뉴런 수와 출력 뉴런 수를 고려해서 적절한 크기의 weight를 초기화한다.
- 초기화의 핵심 목적은 순전파 값과 역전파 gradient가 너무 커지거나 작아지는 것을 막는 것이다.